<a href="https://colab.research.google.com/github/techasit239/DADS7202_PigPicture/blob/main/FreshCheck_Phase2_Exp2_Florence2_SAM3_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FreshCheck Phase 2 — Exp 2: Florence-2 → SAM 3

**Step 1 · Experiment 2: 2-stage pipeline — open-vocab detection then refined segmentation**

> 🔗 **ต้องรัน Phase 2 Foundation ก่อน** + **แนะนำให้รัน Exp 1 (SAM 3) ก่อน** เพราะใช้ SAM 3 model เดิม

---

## ทำไมต้องทำ Exp 2?

SAM 3 + text prompt อาจ "หลงไปกินป้ายราคา/ฉลาก/พลาสติก" ได้ — เพราะ SAM 3 เป็น object-centric model อาจตีความ "raw meat" รวมทั้ง tray ที่ใส่เนื้อ

**Solution:** ใช้ 2-stage pipeline ตามที่อาจารย์แนะนำ:
1. **Florence-2** detect bounding boxes ของ "meat tissue" — ได้พื้นที่ candidate
2. **SAM 3** refine แต่ละ box → ได้ mask ที่ละเอียดและไม่หลง

```
   Input image
        │
        ▼
   ┌─────────────────────────┐
   │  Florence-2 (OD task)   │  ← open-vocab detection
   │  prompt: "meat tissue"  │
   └─────────────────────────┘
        │
        ▼  boxes: [(x1,y1,x2,y2), ...]
        │
   ┌─────────────────────────┐
   │  SAM 3 (box prompts)    │  ← refine boundary
   └─────────────────────────┘
        │
        ▼
   Mask
```

## เวลา

| ขั้น | งาน | เวลา |
|---|---|---|
| 2.0 | Setup + auth | 1 นาที |
| 2.1 | Load Florence-2 + SAM 3 | 5-8 นาที |
| 2.2 | Sanity check 1 รูป | 1 นาที |
| 2.3 | ลอง 2-3 prompts บน subset | 5-10 นาที |
| 2.4 | Full eval 150 รูป | ~20-30 นาที |
| 2.5 | Save results | 1 นาที |

**⏱ รวม ~40-50 นาที** (T4 GPU)

---

## ⚠️ ก่อนเริ่ม

1. ทำ **Phase 2 Foundation** + **Exp 1 (SAM 3)** เรียบร้อยแล้ว
2. มี HuggingFace token + SAM 3 access (เหมือน Exp 1)
3. Florence-2 ไม่ gated — ใช้ได้เลย (`florence-community/Florence-2-base`)


## 2.0 — Setup + Authenticate

In [ ]:
# 2.0.1 — Imports + paths
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/FreshCheck'
PHASE2_DIR     = f'{PROJECT_ROOT}/phase2'
THAI_TEST_CSV  = f'{PHASE2_DIR}/thai_test_set.csv'
GT_MASKS_DIR   = f'{PHASE2_DIR}/gt_masks'
CONFIGS_DIR    = f'{PROJECT_ROOT}/configs'

EXP2_DIR       = f'{PHASE2_DIR}/exp2_florence2_sam3'
EXP2_MASKS_DIR = f'{EXP2_DIR}/predicted_masks'
EXP2_RESULTS   = f'{EXP2_DIR}/results.json'
os.makedirs(EXP2_DIR, exist_ok=True)
os.makedirs(EXP2_MASKS_DIR, exist_ok=True)

assert os.path.exists(THAI_TEST_CSV), 'รัน Phase 2 Foundation ก่อน'

sys.path.insert(0, CONFIGS_DIR)
from seg_metrics import compute_iou, compute_dice, evaluate_batch
from seg_viz     import show_seg_comparison, plot_iou_distribution
print('[OK] Paths + utils ready')


In [ ]:
# 2.0.2 — HuggingFace login (for SAM 3)
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print('[OK] Logged in via Colab Secret')
except Exception:
    login()  # interactive


## 2.1 — Load Florence-2 + SAM 3

**Florence-2:** Microsoft's open-vocab vision model — small (230M params), fast, ไม่ gated
**SAM 3:** ตัวเดียวกับ Exp 1 — แต่ครั้งนี้ใช้ box prompt แทน text prompt


In [ ]:
# 2.1.1 — Install + load Florence-2
!pip install -q -U transformers accelerate
import transformers
print(f'transformers: {transformers.__version__}\n')

from transformers import AutoProcessor, Florence2ForConditionalGeneration

FLORENCE_ID = 'florence-community/Florence-2-base'
print(f'Loading Florence-2 ({FLORENCE_ID})...')

florence_processor = AutoProcessor.from_pretrained(FLORENCE_ID)
florence_model = Florence2ForConditionalGeneration.from_pretrained(
    FLORENCE_ID, dtype=torch.float16
).to(device).eval()

print(f'[OK] Florence-2 loaded ({sum(p.numel() for p in florence_model.parameters())/1e6:.0f}M params)')


In [ ]:
# 2.1.2 — Load SAM 3 (gated)
from transformers import Sam3Processor, Sam3Model

SAM3_ID = 'facebook/sam3'
print(f'Loading SAM 3 ({SAM3_ID})...')

sam3_processor = Sam3Processor.from_pretrained(SAM3_ID)
sam3_model = Sam3Model.from_pretrained(SAM3_ID, dtype=torch.float16).to(device).eval()

print(f'[OK] SAM 3 loaded ({sum(p.numel() for p in sam3_model.parameters())/1e6:.0f}M params)')


## 2.2 — Sanity Check on 1 Image

รันทั้ง pipeline บนรูปเดียวเพื่อดูว่า:
1. Florence-2 ให้ box ที่ครอบเนื้อจริง
2. SAM 3 + box → mask สมเหตุสมผล


In [ ]:
# 2.2.1 — Pipeline functions
def detect_with_florence2(image, text_query, threshold=0.3):
    """
    Run Florence-2 open-vocab object detection.
    Returns: list of (x1, y1, x2, y2) bounding boxes in original image coords.
    """
    # Florence-2 task for grounded open-vocab detection
    prompt = f'<OPEN_VOCABULARY_DETECTION>{text_query}'
    inputs = florence_processor(text=prompt, images=image, return_tensors='pt').to(device, torch.float16)

    with torch.no_grad():
        generated_ids = florence_model.generate(
            **inputs, max_new_tokens=512, num_beams=3, do_sample=False
        )
    generated_text = florence_processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed = florence_processor.post_process_generation(
        generated_text,
        task='<OPEN_VOCABULARY_DETECTION>',
        image_size=(image.width, image.height),
    )

    # parsed format: {'<OPEN_VOCABULARY_DETECTION>': {'bboxes': [[x1,y1,x2,y2]], 'bboxes_labels': [...]}}
    result = parsed.get('<OPEN_VOCABULARY_DETECTION>', {})
    return result.get('bboxes', [])


def segment_with_sam3_boxes(image, boxes):
    """
    Use SAM 3 with box prompts (visual prompts).
    Returns: union binary mask (HxW uint8 0/255).
    """
    if not boxes:
        return np.zeros((image.height, image.width), dtype=np.uint8)

    # SAM 3 accepts box prompts via input_boxes parameter
    inputs = sam3_processor(
        images=image,
        input_boxes=[[box] for box in boxes],  # batch of boxes (1 box per query)
        return_tensors='pt',
    ).to(device, torch.float16)

    with torch.no_grad():
        outputs = sam3_model(**inputs)

    pred_masks = sam3_processor.post_process_masks(
        outputs.pred_masks,
        original_sizes=[image.size[::-1]],
        binarize=False,
    )[0]

    # Take union of all box predictions
    union = (pred_masks.sigmoid() > 0.5).any(dim=0).cpu().numpy().astype(np.uint8) * 255
    return union


def run_pipeline(image, text_query):
    """Florence-2 → SAM 3 full pipeline."""
    boxes = detect_with_florence2(image, text_query)
    if not boxes:
        return np.zeros((image.height, image.width), dtype=np.uint8), boxes
    mask = segment_with_sam3_boxes(image, boxes)
    return mask, boxes


# Test on sample
df = pd.read_csv(THAI_TEST_CSV)
sample = df.iloc[0]
img = Image.open(sample['image_path']).convert('RGB')
gt  = np.array(Image.open(sample['mask_path']).convert('L'))

test_query = 'meat tissue'
t0 = time.time()
pred, boxes = run_pipeline(img, test_query)
elapsed = time.time() - t0

print(f'Sample: {sample["filename"]}')
print(f'Query:  "{test_query}"')
print(f'  Boxes detected:  {len(boxes)}')
print(f'  Pipeline time:   {elapsed:.2f}s')
print(f'  IoU:  {compute_iou(pred, gt):.3f}')
print(f'  Dice: {compute_dice(pred, gt):.3f}')


In [ ]:
# 2.2.2 — Visualize sanity check
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img); axes[0].set_title('Image + Florence-2 boxes'); axes[0].axis('off')
for box in boxes:
    x1, y1, x2, y2 = box
    axes[0].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                      fill=False, edgecolor='red', linewidth=2))
axes[1].imshow(gt, cmap='gray');   axes[1].set_title('GT mask');     axes[1].axis('off')
axes[2].imshow(pred, cmap='gray'); axes[2].set_title('Predicted');   axes[2].axis('off')
plt.suptitle(f'Florence-2 → SAM 3 sanity check', fontsize=12)
plt.tight_layout()
plt.show()


## 2.3 — Try 2 Open-Vocab Queries on Subset

ลอง 2 query ที่ความเฉพาะเจาะจงต่างกัน — Florence-2 ตอบสนองต่อ wording ต่างกันบ้าง


In [ ]:
# 2.3.1 — Run on stratified subset
QUERIES = [
    'meat tissue',         # specific anatomical term
    'raw pork',            # specific food term
]

subset_df = df.groupby('class').sample(n=5, random_state=SEED).reset_index(drop=True)
print(f'Subset: {len(subset_df)} images\n')

subset_results = {}
for query in QUERIES:
    print(f'Query: "{query}"')
    ious, dices, n_boxes_list = [], [], []
    for _, row in subset_df.iterrows():
        img = Image.open(row['image_path']).convert('RGB')
        gt  = np.array(Image.open(row['mask_path']).convert('L'))
        pred, boxes = run_pipeline(img, query)
        ious.append(compute_iou(pred, gt))
        dices.append(compute_dice(pred, gt))
        n_boxes_list.append(len(boxes))
    subset_results[query] = {
        'iou_mean': np.mean(ious), 'dice_mean': np.mean(dices),
        'avg_boxes': np.mean(n_boxes_list),
    }
    print(f'  IoU:  {np.mean(ious):.3f}')
    print(f'  Dice: {np.mean(dices):.3f}')
    print(f'  Avg boxes/image: {np.mean(n_boxes_list):.1f}\n')

BEST_QUERY = max(subset_results, key=lambda q: subset_results[q]['iou_mean'])
print(f'★ Best query: "{BEST_QUERY}" (IoU={subset_results[BEST_QUERY]["iou_mean"]:.3f})')


## 2.4 — Full Evaluation

In [ ]:
# 2.4.1 — Run full pipeline on all 150 images
from tqdm.auto import tqdm

print(f'Running Florence-2 → SAM 3 with query "{BEST_QUERY}" on {len(df)} images...')

pred_masks_list = []
gt_masks_list = []
per_sample = []
inference_times = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Florence-2→SAM 3'):
    img = Image.open(row['image_path']).convert('RGB')
    gt  = np.array(Image.open(row['mask_path']).convert('L'))

    t0 = time.time()
    pred, boxes = run_pipeline(img, BEST_QUERY)
    inference_times.append(time.time() - t0)

    Image.fromarray(pred).save(f'{EXP2_MASKS_DIR}/{Path(row["filename"]).stem}_pred.png')

    pred_masks_list.append(pred)
    gt_masks_list.append(gt)
    per_sample.append({
        'filename': row['filename'], 'class': row['class'], 'source': row['source'],
        'iou':  compute_iou(pred, gt),
        'dice': compute_dice(pred, gt),
        'n_boxes': len(boxes),
    })

print(f'\n[OK] Done — avg time: {np.mean(inference_times):.2f}s/image')


In [ ]:
# 2.4.2 — Aggregate metrics
results = evaluate_batch(pred_masks_list, gt_masks_list)
per_sample_df = pd.DataFrame(per_sample)

print(f'OVERALL (n={results["n_samples"]}):')
print(f'  Mean IoU:  {results["mean_iou"]:.4f} ± {results["std_iou"]:.4f}')
print(f'  Mean Dice: {results["mean_dice"]:.4f} ± {results["std_dice"]:.4f}')

print(f'\nPer class:')
for cls in per_sample_df['class'].unique():
    sub = per_sample_df[per_sample_df['class'] == cls]
    print(f'  {cls:12s}: IoU={sub["iou"].mean():.3f}, Dice={sub["dice"].mean():.3f}')

print(f'\nPer source:')
for src in per_sample_df['source'].unique():
    sub = per_sample_df[per_sample_df['source'] == src]
    print(f'  {src:12s}: IoU={sub["iou"].mean():.3f}, Dice={sub["dice"].mean():.3f}')

# Detection stats
no_detection = (per_sample_df['n_boxes'] == 0).sum()
print(f'\nImages with NO boxes detected: {no_detection} ({no_detection/len(per_sample_df)*100:.1f}%)')


In [ ]:
# 2.4.3 — Plot + visualize
plot_iou_distribution(results, figsize=(12, 4))
plt.suptitle(f'Exp 2: Florence-2 → SAM 3 — IoU/Dice Distribution', fontsize=12)
plt.tight_layout()
plt.savefig(f'{EXP2_DIR}/iou_dice_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

# Show worst/median/best
sorted_idx = np.argsort(per_sample_df['iou'].values)
for label, idx in [('Worst', sorted_idx[0]),
                    ('Median', sorted_idx[len(sorted_idx) // 2]),
                    ('Best', sorted_idx[-1])]:
    row = df.iloc[idx]
    fig = show_seg_comparison(
        row['image_path'], gt_masks_list[idx], pred_masks_list[idx],
        title=f'{label} · {row["filename"][:30]} · IoU={per_sample[idx]["iou"]:.3f}'
    )
    plt.show()


## 2.5 — Save Results

In [ ]:
# 2.5.1 — Save JSON + CSV
final = {
    'experiment': 'Phase 2 Exp 2 — Florence-2 → SAM 3',
    'florence_model': FLORENCE_ID,
    'sam3_model': SAM3_ID,
    'best_query': BEST_QUERY,
    'all_queries_tried': subset_results,
    'config': {
        'n_test_samples': len(df),
        'florence_detection_threshold': 0.3,
        'sam3_mask_threshold': 0.5,
        'seed': SEED,
    },
    'aggregate': {
        'mean_iou':  results['mean_iou'],  'std_iou':  results['std_iou'],
        'mean_dice': results['mean_dice'], 'std_dice': results['std_dice'],
        'n_samples': results['n_samples'],
    },
    'per_class': {
        cls: {
            'iou_mean':  float(per_sample_df[per_sample_df['class'] == cls]['iou'].mean()),
            'dice_mean': float(per_sample_df[per_sample_df['class'] == cls]['dice'].mean()),
        } for cls in per_sample_df['class'].unique()
    },
    'per_source': {
        src: {
            'iou_mean':  float(per_sample_df[per_sample_df['source'] == src]['iou'].mean()),
            'dice_mean': float(per_sample_df[per_sample_df['source'] == src]['dice'].mean()),
        } for src in per_sample_df['source'].unique()
    },
    'no_detection_count': int((per_sample_df['n_boxes'] == 0).sum()),
    'avg_inference_time_sec': float(np.mean(inference_times)),
    'output_dirs': {'predicted_masks': EXP2_MASKS_DIR},
}

with open(EXP2_RESULTS, 'w') as f:
    json.dump(final, f, indent=2, ensure_ascii=False)
per_sample_df.to_csv(f'{EXP2_DIR}/per_sample_metrics.csv', index=False)

print(f'[OK] Saved {EXP2_RESULTS}')


In [ ]:
# 2.5.2 — Final summary
print('=' * 60)
print('PHASE 2 — Exp 2 (Florence-2 → SAM 3) — DONE')
print('=' * 60)
print(f'\nBest query:  "{BEST_QUERY}"')
print(f'Test samples: {results["n_samples"]}')
print(f'\nResults on Thai test set:')
print(f'  Mean IoU:  {results["mean_iou"]:.4f} ± {results["std_iou"]:.4f}')
print(f'  Mean Dice: {results["mean_dice"]:.4f} ± {results["std_dice"]:.4f}')
print(f'\nAverage inference time: {np.mean(inference_times):.2f}s/image')
print(f'\n[OK] Ready for comparison with Exp 1 + Exp 3')
